# Waste generation prediction — ML baseline

Features and target chosen in `data_exploration.ipynb`. Population dominates linearly (r=0.89 with target); this notebook checks whether tree ensembles can do better than a population-only model by also using composition/income/region.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

df = pd.read_csv("../data/WhatAWaste_City_Level.csv")

target = "total_msw_total_msw_generated_tons_year"
df[target] = pd.to_numeric(df[target].astype(str).str.replace(",", "", regex=False), errors="coerce")

features = [
    "population_population_number_of_people",
    "income_id",
    "region_id",
    "composition_food_organic_waste_percent",
    "composition_paper_cardboard_percent",
    "composition_plastic_percent",
    "composition_glass_percent",
    "composition_metal_percent",
    "composition_other_percent",
]

data = df[features + [target]].dropna(subset=[target]).copy()
data.shape

(326, 10)

## Feature prep

- log-transform population and target — both are heavily right-skewed (a handful of megacities vs. hundreds of smaller cities)
- one-hot encode `income_id` / `region_id`
- median-impute the composition percentages (still missing in 20-30% of rows)

In [2]:
data["log_population"] = np.log1p(data["population_population_number_of_people"])
data["log_target"] = np.log1p(data[target])

composition_cols = [c for c in features if c.startswith("composition_")]
imputer = SimpleImputer(strategy="median")
data[composition_cols] = imputer.fit_transform(data[composition_cols])

data = pd.get_dummies(data, columns=["income_id", "region_id"], drop_first=True)

feature_cols = [c for c in data.columns if c not in (target, "log_target", "population_population_number_of_people")]
X = data[feature_cols]
y = data["log_target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((260, 16), (66, 16))

In [3]:
def evaluate(name, model, X_test, y_test):
    pred_log = model.predict(X_test)
    pred = np.expm1(pred_log)
    actual = np.expm1(y_test)
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    print(f"{name}: MAE={mae:,.0f} tons, RMSE={rmse:,.0f} tons, R2={r2:.3f}")
    return {"model": name, "mae": mae, "rmse": rmse, "r2": r2}

results = []

In [4]:
rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
results.append(evaluate("Random Forest", rf, X_test, y_test))

Random Forest: MAE=245,688 tons, RMSE=470,268 tons, R2=0.804


In [5]:
xgb = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
results.append(evaluate("XGBoost", xgb, X_test, y_test))

XGBoost: MAE=240,257 tons, RMSE=461,493 tons, R2=0.811


## Population-only baseline

Worth checking explicitly: does adding composition/income/region actually beat just using population?

In [6]:
rf_pop_only = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_pop_only.fit(X_train[["log_population"]], y_train)
results.append(evaluate("Random Forest (population only)", rf_pop_only, X_test[["log_population"]], y_test))

Random Forest (population only): MAE=331,001 tons, RMSE=619,955 tons, R2=0.660


In [7]:
pd.DataFrame(results).sort_values("r2", ascending=False)

,model,mae,rmse,r2
1,XGBoost,240256.507617,461493.100368,0.811323
0,Random Forest,245687.629688,470268.239684,0.804079
2,Random Forest (population only),331000.673845,619955.371616,0.659506


In [8]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(10)

log_population                            0.923369
composition_metal_percent                 0.013597
composition_food_organic_waste_percent    0.011711
composition_paper_cardboard_percent       0.008727
composition_glass_percent                 0.007890
composition_other_percent                 0.007803
composition_plastic_percent               0.007373
region_id_SAS                             0.006247
region_id_SSF                             0.004019
income_id_LIC                             0.002228
dtype: float64

## Conclusion

Population is by far the dominant signal — the full feature set barely beats the population-only model. That matches the EDA correlation finding. The main thing tree ensembles add here is the log-transform handling of population's skew, not the extra composition/income/region features.

Next: `01_deep_learning.ipynb` tries an MLP on the same features to see whether it can pick up nonlinear interactions (e.g. income level changing how composition affects total waste) that these ML models aren't using.